In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv(r"C:\Programming\Machine Learning\Used Car Price Intelligence\Used Car Arbitrage\data\Processed\clean_car_listing.csv")

df = df.dropna()

y = df['Clean_Price']

X = df.drop(columns=["Clean_Price"])

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2, random_state=42)

print(f"Training data Shape: {X_train.shape}")
print(f"Testing data Shape: {X_test.shape}")
X_train.head()

Training data Shape: (400, 6)
Testing data Shape: (100, 6)


,Fuel_Type,Transmission,Clean_Kilometers,Year,Brand,Model
249,Petrol,Missing Transmission,16624,2023,Tata,NEXON
433,Petrol,Manual,29957,2021,Hyundai,NEW I20
19,CNG,Manual,127453,2016,Hyundai,Grand i10
322,Petrol,Missing Transmission,27300,2021,Hyundai,GRAND I10 NIOS
332,Petrol,Missing Transmission,32313,2019,Honda,Amaze


In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_features = ['Year', 'Clean_Kilometers']
Categorical_features = ['Brand','Model','Fuel_Type','Transmission']
numeric_transformer = Pipeline(steps=[("Scaler",StandardScaler())])
# building rules for text -- Convert Categories (brand, model, transmission,fuel-type) into binary (0,1)
Categorical_Transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore',sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num',numeric_transformer,numeric_features),
    ('category',Categorical_Transformer,Categorical_features)
])

print("Data Preprocessor successfully built!")


Data Preprocessor successfully built!


In [9]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor,GradientBoostingRegressor
from sklearn.metrics import r2_score,mean_absolute_error,mean_squared_error

models = {
    "Linear Regression ":LinearRegression(),
    "Random Forest ": RandomForestRegressor(random_state=42),
    "Gradient Boost ":GradientBoostingRegressor(random_state=42)
}

trained_pipelines = {}

for name,model in models.items():
    pipeline = Pipeline(steps=[
        ('preprocessor',preprocessor),
        ('regressor',model)
    ])

    pipeline.fit(X_train,y_train)

    trained_pipelines[name]=pipeline

    predictions = pipeline.predict(X_test)

    mae = mean_absolute_error(y_test,predictions)
    rmse = np.sqrt(mean_squared_error(y_test,predictions))
    r2 = r2_score(y_test,predictions)

    print(f"-------{name}-------")
    print(f"Mean Absolute Error :₹{int(mae):,}")
    print(f"Root Mean Squared Error : ₹{int(rmse):,}")
    print(f"R2 Score: {r2}\n") 

-------Linear Regression -------
Mean Absolute Error :₹71,709
Root Mean Squared Error : ₹111,027
R2 Score: 0.744394675511903

-------Random Forest -------
Mean Absolute Error :₹66,517
Root Mean Squared Error : ₹130,440
R2 Score: 0.6471939477587276

-------Gradient Boost -------
Mean Absolute Error :₹70,662
Root Mean Squared Error : ₹126,300
R2 Score: 0.6692324548773743



In [11]:
import shap
import matplotlib.pyplot as plt 

best_pipeline = trained_pipelines['Random Forest ']
rf_model = best_pipeline.named_steps['regressor']
preprocessor = best_pipeline.named_steps['preprocessor']

X_test_transformed = preprocessor.transform(X_test)
feature_names = preprocessor.get_feature_names_out()

explainer = shap.TreeExplainer(rf_model)
shap_values = explainer.shap_values(X_test_transformed)

plt.figure(figsize=(10,6))
shap.summary_plot(shap_values,X_test_transformed,feature_names=feature_names)

ImportError: matplotlib is not installed so plotting is not available! Run `pip install matplotlib` to fix this.

<Figure size 1000x600 with 0 Axes>